# Rightsize quickstart

Pick a model, pick your hardware, get a runnable plan. This notebook runs the whole first slice:
detect the machine, predict memory and speed for a few GGUF quants, then (optionally) produce and
evaluate the files.

Setup once from the repo root:

```bash
uv sync --group dev --extra llamacpp      # torch CPU + transformers for conversion
# llama.cpp binaries: download a release into .tools/llama.cpp (see docs/guide/choosing-a-model-format.md)
```

In [ ]:
import rightsize

rightsize.__version__

## 1. What machine is this?

In [ ]:
from rightsize.hardware import detect

dev = detect()
dev

## 2. What does the model need? (no download; reads Hub headers only)

In [ ]:
from rightsize.catalog import facts

fx = facts("Qwen/Qwen3-1.7B")
fx.params_total / 1e9, fx.num_layers, fx.num_kv_heads, fx.head_dim, fx.dtype

In [ ]:
from rightsize.fit import estimate, predicted_file_gb

for q in ["Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]:
    r = estimate(fx, q, dev, ctx=8192)
    print(
        f"{q:7s} file {predicted_file_gb(fx, q):.2f} GB  vram {r.vram_gb:.2f} GB  {r.verdict.value:7s}  {r.speed} {r.speed_unit}"
    )

## 3. Make the files (needs llama.cpp and the `llamacpp` extra)

`quantize_model` renders each step from the recipe registry, runs it, and records what it measured
next to what it predicted. Start with a dry run to see the commands.

In [ ]:
from rightsize.execution import quantize_model

m = quantize_model("Qwen/Qwen3-1.7B", ["Q4_K_M"], dry_run=True)
[s.recipe_id for s in m.steps]

In [ ]:
# Real run: downloads ~4 GB, converts on CPU, quantizes, then measures KL divergence vs the 16-bit file.
# m = quantize_model("Qwen/Qwen3-1.7B", ["Q4_K_M", "Q8_0"], imatrix=True, evaluate=True, eval_chunks=50)
# m.gate, m.artifacts

Same thing from the shell: `rightsize quantize Qwen/Qwen3-1.7B --quant Q4_K_M --imatrix --eval`.